In [3]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # looks for .env from the notebook's folder upwards


True

In [5]:
from google import genai
from google.genai import types

client = genai.Client(http_options=types.HttpOptions(timeout=30000)) 
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/gemini-4-neon
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/deep-research-pro-preview-12-2025


In [2]:
from google import genai
import time
from google.genai import types, errors

#client = genai.Client()  # reads GEMINI_API_KEY
client = genai.Client(http_options=types.HttpOptions(timeout=30000)) 
MODEL = "gemini-3.6-flash"
CONTENT = "Explain in two sentences what a token is."

#resp = client.models.generate_content(
#    model=MODEL,
#    contents="Explain in two sentences what a token is.",
#    config=types.GenerateContentConfig(max_output_tokens=1000),
#)

def generate(model, prompt, retries=4):
    for i in range(retries):
        try:
            return client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    max_output_tokens=100,
                    thinking_config=types.ThinkingConfig(
                        thinking_level="minimal"
                    )
                ),
            )
        except errors.ServerError:
            time.sleep(2 ** i)  # 1s, 2s, 4s, 8s
    raise RuntimeError("Model still unavailable")
resp = generate(MODEL, CONTENT)
print(resp.text)
print(resp.usage_metadata)                # prompt_token_count, candidates_token_count, thoughts_token_count
print(resp.candidates[0].finish_reason)   # STOP if it finished properly

In processing natural language, a token is a basic unit of text—such as a word, sub-word, or punctuation mark—that an AI model uses to analyze and generate human language. It serves as the fundamental building block that allows algorithms to break down, quantify, and understand text data.
cache_tokens_details=None cached_content_token_count=None candidates_token_count=60 candidates_tokens_details=None prompt_token_count=10 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=10
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=70 traffic_type=None
FinishReason.STOP


In [3]:
texts = {
    "es": "La factura se emite al cerrar el pedido y se envía por correo al cliente.",
    "en": "The invoice is issued when the order is closed and emailed to the customer.",
}
for lang, text in texts.items():
    r = client.models.count_tokens(model=MODEL, contents=text)
    print(lang, len(text.split()), "words ->", r.total_tokens, "tokens")

es 15 words -> 18 tokens
en 14 words -> 16 tokens


In [4]:
texts = {
    "es": "Feyre Archeron is a 19-year-old human huntress who keeps her poor family alive. In the woods she kills a wolf, which turns out to be a faerie in disguise. As punishment, a beast-like High Fae named Tamlin takes her to Prythian, the faerie realm beyond the wall that separates it from the human lands. There she lives at his estate in the Spring Court and slowly falls in love with him. She learns that a curse is spreading across the land, and that the cruel queen Amarantha holds Tamlin’s court and all of Prythian under her power. To save him, Feyre goes Under the Mountain and faces deadly trials. Rhysand, the High Lord of the Night Court, helps her along the way. In the end she breaks the curse, but only after dying and being reborn as High Fae.",
    "en": "Feyre Archeron es una cazadora humana de 19 años que mantiene con vida a su pobre familia. En el bosque mata a un lobo, que resulta ser un hada disfrazada. Como castigo, Tamlin, un Alto Fae con aspecto de bestia, se la lleva a Prythian, el reino de los feéricos al otro lado del muro que lo separa del mundo humano. Allí vive en su mansión de la Corte de la Primavera y poco a poco se enamora de él. Descubre que una maldición se extiende por la tierra y que la cruel reina Amarantha tiene sometidos a Prythian y a la corte de Tamlin. Para salvarlo, Feyre baja a Bajo la Montaña y se enfrenta a pruebas mortales, con la ayuda de Rhysand, Alto Lord de la Corte de la Noche. Al final rompe la maldición, pero solo después de morir y renacer como Alta Fae.",
}
for lang, text in texts.items():
    r = client.models.count_tokens(model=MODEL, contents=text)
    print(lang, len(text.split()), "words ->", r.total_tokens, "tokens")

es 139 words -> 184 tokens
en 147 words -> 200 tokens


In [4]:
import anthropic
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
MODEL = "claude-haiku-5-5"
msg = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": "Explain in two sentences what a token is."}],
)
print(msg.content[0].text)
print(msg.usage)                        # input_tokens, output_tokens
print(msg.stop_reason)                  # "end_turn" if it finished properly

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChEh2v3BZQX9d73oHD3Aj'}

In [6]:
texts = {
    "es": "Feyre Archeron is a 19-year-old human huntress who keeps her poor family alive. In the woods she kills a wolf, which turns out to be a faerie in disguise. As punishment, a beast-like High Fae named Tamlin takes her to Prythian, the faerie realm beyond the wall that separates it from the human lands. There she lives at his estate in the Spring Court and slowly falls in love with him. She learns that a curse is spreading across the land, and that the cruel queen Amarantha holds Tamlin’s court and all of Prythian under her power. To save him, Feyre goes Under the Mountain and faces deadly trials. Rhysand, the High Lord of the Night Court, helps her along the way. In the end she breaks the curse, but only after dying and being reborn as High Fae.",
    "en": "Feyre Archeron es una cazadora humana de 19 años que mantiene con vida a su pobre familia. En el bosque mata a un lobo, que resulta ser un hada disfrazada. Como castigo, Tamlin, un Alto Fae con aspecto de bestia, se la lleva a Prythian, el reino de los feéricos al otro lado del muro que lo separa del mundo humano. Allí vive en su mansión de la Corte de la Primavera y poco a poco se enamora de él. Descubre que una maldición se extiende por la tierra y que la cruel reina Amarantha tiene sometidos a Prythian y a la corte de Tamlin. Para salvarlo, Feyre baja a Bajo la Montaña y se enfrenta a pruebas mortales, con la ayuda de Rhysand, Alto Lord de la Corte de la Noche. Al final rompe la maldición, pero solo después de morir y renacer como Alta Fae.",
}
for lang, text in texts.items():
    r = client.messages.count_tokens(
        model=MODEL,
        messages=[{"role": "user", "content": text}],
    )
    print(lang, len(text.split()), "words ->", r.input_tokens, "tokens")

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChEh6xvfxNRc6nfQnRc3C'}